In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling  import SMOTE

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('All libraries ready!')

All libraries ready!


In [2]:
df = pd.read_csv('../data/processed/features.csv')

print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')


X = df.drop('label', axis=1)
y = df['label']

feature_names = X.columns.tolist()

print(f'\nFeatures ({len(feature_names)}): {feature_names}')
print(f'\nLabel distribution:')
print(y.value_counts())
print(f'\nImbalance ratio: {y.value_counts()[0] / y.value_counts()[1]:.2f}:1')

Dataset: 507,195 rows × 23 columns

Features (22): ['url_length', 'domain_length', 'path_length', 'query_length', 'has_at_symbol', 'has_ip_address', 'has_double_slash', 'has_dash_in_domain', 'has_port', 'dot_count', 'hyphen_count', 'param_count', 'special_char_ratio', 'is_https', 'has_suspicious_tld', 'subdomain_depth', 'suspicious_word_count', 'brand_in_subdomain', 'domain_entropy', 'url_entropy', 'digit_ratio', 'letter_ratio']

Label distribution:
label
0    392897
1    114298
Name: count, dtype: int64

Imbalance ratio: 3.44:1


In [3]:


print('=== Before SMOTE ===')
print(y.value_counts())

print('\nApplying SMOTE... (takes 2-3 minutes on 500k rows)')

smote = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X, y)

print('\n=== After SMOTE ===')
print(pd.Series(y_res).value_counts())
print(f'\nNew total size: {len(X_res):,}')
print(f'Now perfectly balanced: 1:1 ratio')

=== Before SMOTE ===
label
0    392897
1    114298
Name: count, dtype: int64

Applying SMOTE... (takes 2-3 minutes on 500k rows)

=== After SMOTE ===
label
1    392897
0    392897
Name: count, dtype: int64

New total size: 785,794
Now perfectly balanced: 1:1 ratio


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_res,
    y_res,
    test_size=0.2,       
    random_state=42,
    stratify=y_res       
)

print(f'Training set : {X_train.shape[0]:,} samples')
print(f'Test set     : {X_test.shape[0]:,} samples')
print(f'Features     : {X_train.shape[1]}')
print(f'\nTrain label split: {pd.Series(y_train).value_counts().to_dict()}')
print(f'Test label split : {pd.Series(y_test).value_counts().to_dict()}')

Training set : 628,635 samples
Test set     : 157,159 samples
Features     : 22

Train label split: {1: 314318, 0: 314317}
Test label split : {0: 78580, 1: 78579}


In [ ]:
print('Training Random Forest...')
print('(uses all CPU cores, should take 1-3 minutes)\n')

rf = RandomForestClassifier(
    n_estimators=100,

    max_depth=None,

    min_samples_split=5,

    class_weight='balanced',


    random_state=42,
    n_jobs=1,
    verbose=0

)

rf.fit(X_train, y_train)
print('Random Forest Model Traied!')

rf_preds= rf.predict(X_test)

rf_proba= rf.predict_proba(X_test)[:,1]
print(f'Sample predictions : {rf_preds[:8]}')
print(f'Sample probabilities: {rf_proba[:8].round(3)}')

Training Random Forest...
(uses all CPU cores, should take 1-3 minutes)

Random Forest Model Traied!
Sample predictions : [1 1 1 1 1 1 1 0]
Sample probabilities: [[0.01 0.99]]


In [13]:
print('Training XGBoost....')

xgb = XGBClassifier(
    n_estimators=200,

    max_depth=6,

    learning_rate=0.1,

    subsample=0.8,

    colsample_bytree=0.8,

    eval_metric='logloss',

    random_state=42,
    n_jobs=1,
    verbosity=0
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print('XGBoost trained!')

xgb_preds=xgb.predict(X_test)
xgb_proba=xgb.predict_proba(X_test)[:,1]

Training XGBoost....
XGBoost trained!


In [14]:
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f'\n{"═" * 55}')
    print(f'  {name}')
    print(f'{"═" * 55}')

    print('\n--- Classification Report ---')
    print(classification_report(
        y_true, y_pred,
        target_names=['Legitimate (0)', 'Phishing (1)']
    ))

    auc = roc_auc_score(y_true, y_proba)
    print(f'AUC-ROC: {auc:.4f}')

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f'\n--- Confusion Matrix Breakdown ---')
    print(f'True Negatives  (safe → predicted safe)    : {tn:,}')
    print(f'False Positives (safe → predicted phishing) : {fp:,}  ← false alarm')
    print(f'False Negatives (phishing → predicted safe) : {fn:,}  ← MISSED = DANGEROUS')
    print(f'True Positives  (phishing → caught!)        : {tp:,}')

    phishing_recall = tp / (tp + fn)
    miss_rate       = fn / (tp + fn)
    print(f'\nPhishing catch rate : {phishing_recall:.2%}')
    print(f'Miss rate           : {miss_rate:.2%}  ← want this as LOW as possible')

    return auc, cm

rf_auc,  rf_cm  = evaluate_model('RANDOM FOREST', y_test, rf_preds,  rf_proba)
xgb_auc, xgb_cm = evaluate_model('XGBOOST',       y_test, xgb_preds, xgb_proba)


═══════════════════════════════════════════════════════
  RANDOM FOREST
═══════════════════════════════════════════════════════

--- Classification Report ---
                precision    recall  f1-score   support

Legitimate (0)       0.93      0.94      0.94     78580
  Phishing (1)       0.94      0.93      0.93     78579

      accuracy                           0.93    157159
     macro avg       0.93      0.93      0.93    157159
  weighted avg       0.93      0.93      0.93    157159



ValueError: Found input variables with inconsistent numbers of samples: [157159, 1]